# Clustering vehicle safety records

**CSCI 5612 - Module 2, Project Part 2**

If the only thing known about a vehicle were its measured safety record - no
make, no model, no body style, no star rating - what groups would the data put
it into on its own?

This notebook runs k-means and hierarchical clustering over 229 vehicles
described by 16 numeric safety measures. The hierarchical clustering uses
**cosine distance**, as the assignment specifies.

In [1]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 5612

## 1. Data preparation

Clustering accepts **only unlabeled numeric data**. Distances cannot be computed
on text, and an unsupervised method that can see a label is no longer
unsupervised. Everything identifying or categorical is removed and held aside
to interpret the result afterwards.

In [2]:
df = pd.read_csv("../data/clean/vehicle_model_year.csv")
print("source table:", df.shape)
df.head(3)

source table: (229, 30)


,make,model,model_year,segment,n_complaints,n_crash,n_fire,total_injuries,total_deaths,median_days_to_report,mean_vehicle_age,pct_severe,crash_rate,fire_rate,injuries_per_100_complaints,top_component,n_recalls,n_park_it_recalls,n_rated_trims,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,InvestigationCount,esc_standard,has_fcw,has_ldw,rollover_risk_band,rating_class
0,audi,a4,2016,Luxury,25,0,2,0,0,14.0,4.0800,0.0800,0.0000,0.08,0.000,ENGINE,0,0,2,5.0,5.0,5.0,5.0,0.099,0,1,0,0,Low (<10%),5-star
1,audi,a4,2017,Luxury,82,4,0,2,0,18.5,3.4512,0.0732,0.0488,0.00,2.439,AIR BAGS,1,0,2,5.0,4.0,5.0,5.0,0.099,0,1,1,0,Low (<10%),5-star
2,audi,a4,2018,Luxury,42,1,0,0,0,73.5,5.1429,0.0238,0.0238,0.00,0.000,ELECTRICAL SYSTEM,1,0,2,5.0,4.0,5.0,5.0,0.099,0,1,1,0,Low (<10%),5-star


In [3]:
FEATURES = [
    "n_complaints", "crash_rate", "fire_rate", "pct_severe",
    "injuries_per_100_complaints", "median_days_to_report", "mean_vehicle_age",
    "n_recalls", "InvestigationCount", "OverallRating",
    "OverallFrontCrashRating", "OverallSideCrashRating", "RolloverRating",
    "RolloverPossibility", "has_fcw", "has_ldw",
]
HELD_OUT = ["make", "model", "model_year", "segment", "top_component",
            "rollover_risk_band", "rating_class"]

held_out = df[HELD_OUT].copy()     # never seen by the algorithms
X = df[FEATURES].copy()

# Raw event counts are excluded on purpose: they scale with complaint volume,
# so keeping them would let fleet size dominate every distance.
# Skewed counts are log-transformed first.
for col in ["n_complaints", "n_recalls", "InvestigationCount"]:
    X[col] = np.log1p(X[col])
    X = X.rename(columns={col: f"log_{col}"})

print("BEFORE standardising — note how unequal the variances are:")
X.describe().loc[["mean", "std", "min", "max"]].round(2)

BEFORE standardising — note how unequal the variances are:


,log_n_complaints,crash_rate,fire_rate,pct_severe,injuries_per_100_complaints,median_days_to_report,mean_vehicle_age,log_n_recalls,log_InvestigationCount,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,has_fcw,has_ldw
mean,5.23,0.07,0.01,0.09,5.31,12.93,2.99,1.47,0.60,4.83,4.56,4.98,4.24,0.13,0.55,0.53
std,1.25,0.06,0.02,0.06,5.31,9.60,1.32,0.82,0.65,0.36,0.51,0.15,0.48,0.04,0.50,0.50
min,1.10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,3.00,4.00,3.00,0.07,0.00,0.00
max,7.80,0.43,0.12,0.43,42.86,84.00,6.66,3.53,2.77,5.00,5.00,5.00,5.00,0.21,1.00,1.00


In [4]:
Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
print("AFTER standardising:", Xs.shape)
Xs.describe().loc[["mean", "std", "min", "max"]].round(2)

AFTER standardising: (229, 16)


,log_n_complaints,crash_rate,fire_rate,pct_severe,injuries_per_100_complaints,median_days_to_report,mean_vehicle_age,log_n_recalls,log_InvestigationCount,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,has_fcw,has_ldw
mean,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.0,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1.00,1.00,1.00
min,-3.30,-1.15,-0.71,-1.43,-1.00,-1.35,-2.26,-1.78,-0.93,-2.30,-3.09,-6.53,-2.6,-1.87,-1.11,-1.07
max,2.06,6.17,5.71,5.50,7.08,7.42,2.78,2.50,3.35,0.48,0.86,0.16,1.6,2.02,0.90,0.94


## 2. k-means, swept across k

The silhouette width measures how much closer each vehicle sits to its own
cluster than to the next nearest one.

In [5]:
K_RANGE = range(2, 16)
rows, models = [], {}
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE)
    labels = km.fit_predict(Xs)
    rows.append({"k": k,
                 "silhouette": silhouette_score(Xs, labels),
                 "inertia": km.inertia_,
                 "smallest_cluster": int(pd.Series(labels).value_counts().min())})
    models[k] = labels
km_sweep = pd.DataFrame(rows)
km_sweep.round(4)

,k,silhouette,inertia,smallest_cluster
0,2,0.1820,3037.8232,99
1,3,0.1759,2685.9568,40
2,4,0.1728,2431.9887,40
3,5,0.1962,2194.0876,6
4,6,0.2138,2025.8141,5
5,7,0.2253,1869.0500,5
6,8,0.2163,1740.0434,5
7,9,0.2140,1629.1685,5
8,10,0.2293,1540.8690,3
9,11,0.2044,1474.5169,4


**The naive answer is wrong here.** Taking the highest silhouette would return
k=15 - the edge of the tested range, where the smallest cluster holds three
vehicles. A curve that keeps rising with k is the signature of a dataset with
no clean separation: the algorithm is slicing a continuum into ever smaller
pieces, and smaller pieces always look tidier.

Only **interior local maxima** are treated as candidates.

In [6]:
s, ks = km_sweep["silhouette"].values, km_sweep["k"].values
local_max = [int(ks[i]) for i in range(1, len(s) - 1)
             if s[i] > s[i - 1] and s[i] > s[i + 1]]
print("interior local maxima:", local_max)

interior local maxima: [7, 10, 13]


## 3. Hierarchical clustering with cosine distance

Cosine distance is `1 - cosine similarity`. It compares the *shape* of two
records regardless of magnitude, so a heavily complained-about model and a
lightly complained-about one group together if their problems are
proportionally alike.

Ward linkage is **not** available: it is defined only for squared Euclidean
distance. Average linkage is the standard pairing with cosine.

In [7]:
dist = pdist(Xs.values, metric="cosine")
Z = linkage(dist, method="average")

rows = []
for k in K_RANGE:
    labels = fcluster(Z, t=k, criterion="maxclust")
    sizes = pd.Series(labels).value_counts()
    rows.append({"k": k,
                 "silhouette": silhouette_score(squareform(dist), labels,
                                                metric="precomputed"),
                 "smallest_cluster": int(sizes.min()),
                 "n_singletons": int((sizes == 1).sum())})
hc_sweep = pd.DataFrame(rows)
hc_sweep.round(4)

,k,silhouette,smallest_cluster,n_singletons
0,2,0.2873,113,0
1,3,0.2803,43,0
2,4,0.2540,16,0
3,5,0.2630,16,0
4,6,0.2795,16,0
5,7,0.3035,6,0
6,8,0.2997,6,0
7,9,0.2904,4,0
8,10,0.2897,4,0
9,11,0.2755,4,0


A cluster containing one vehicle has a silhouette width of **1 by definition**,
so any k that strands singletons is rewarded for isolating outliers rather than
for finding structure. Those solutions are discarded before reading the peak.

In [8]:
usable = hc_sweep[hc_sweep["n_singletons"] == 0]
sil_k = int(usable.loc[usable["silhouette"].idxmax(), "k"])
hc = hc_sweep.set_index("k")["silhouette"]
best_k = max(local_max, key=lambda k: hc.get(k, -1))

print("hierarchical peak (no singletons):     k =", sil_k)
print("k-means local maxima scored by hclust:",
      {k: round(float(hc[k]), 4) for k in local_max})
print("\n>>> both methods agree on k =", best_k)

hierarchical peak (no singletons):     k = 7
k-means local maxima scored by hclust: {7: 0.3035, 10: 0.2897, 13: 0.2698}

>>> both methods agree on k = 7


## 4. Do the two methods agree?

They use different distance metrics and entirely different procedures, so
agreement is meaningful evidence.

In [9]:
km_labels = models[best_k]
hc_labels = fcluster(Z, t=sil_k, criterion="maxclust")
ari = adjusted_rand_score(km_labels, hc_labels)
print(f"adjusted Rand index: {ari:.3f}")
pd.crosstab(pd.Series(km_labels + 1, name="k-means"),
            pd.Series(hc_labels, name="hierarchical"))

adjusted Rand index: 0.638


hierarchical,1,2,3,4,5,6,7
k-means,,,,,,,
1,10,0,0,0,0,0,0
2,3,0,0,1,0,56,3
3,0,0,0,0,35,1,0
4,1,1,0,0,0,0,12
5,21,12,42,0,0,0,0
6,0,23,0,0,2,0,1
7,0,0,0,5,0,0,0


## 5. What the clusters contain

Segment is checked **only now**, after fitting.

In [10]:
out = held_out.copy()
out["kmeans_cluster"] = km_labels + 1
pd.crosstab(out["kmeans_cluster"], out["segment"])

segment,Electric,Hybrid,Luxury,Minivan,SUV,Sedan,Truck
kmeans_cluster,,,,,,,
1,7,0,0,0,1,2,0
2,1,1,6,12,23,20,0
3,0,0,0,0,21,0,15
4,1,0,6,1,4,2,0
5,1,2,2,9,34,27,0
6,0,6,2,0,3,13,2
7,1,0,0,0,0,4,0


In [11]:
profile = df[FEATURES].copy()
profile["cluster"] = km_labels + 1
profile.groupby("cluster").mean().round(3)

,n_complaints,crash_rate,fire_rate,pct_severe,injuries_per_100_complaints,median_days_to_report,mean_vehicle_age,n_recalls,InvestigationCount,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,has_fcw,has_ldw
cluster,,,,,,,,,,,,,,,,
1,796.300,0.088,0.004,0.103,5.467,7.750,2.826,18.300,10.200,5.000,5.000,5.000,4.900,0.081,0.900,1.000
2,602.524,0.038,0.012,0.056,3.173,12.524,4.144,5.635,1.651,4.984,4.807,4.992,4.206,0.129,0.032,0.032
3,401.250,0.057,0.013,0.077,4.645,12.875,3.269,6.639,1.222,4.132,4.028,5.000,3.819,0.181,0.361,0.306
4,116.286,0.072,0.065,0.141,6.388,22.750,3.561,1.857,0.929,5.000,4.286,5.000,4.571,0.120,0.500,0.286
5,210.040,0.054,0.008,0.071,4.248,11.260,2.087,3.973,0.467,4.993,4.700,5.000,4.293,0.135,1.000,0.987
6,86.962,0.189,0.013,0.208,14.533,15.692,2.042,1.615,0.577,4.923,4.269,5.000,4.346,0.115,0.769,0.808
7,121.200,0.032,0.009,0.044,1.886,11.800,3.482,1.800,0.400,4.000,4.800,4.000,4.000,0.104,0.000,0.000


## 6. What this says about the topic

Silhouette widths peak near 0.3, where above 0.5 would indicate clean
separation. Vehicle safety records form a **continuum, not a set of distinct
types** - there is no natural population of dangerous cars sitting apart from
safe ones. That is a substantive finding, and it is part of why a five-point
star rating struggles to separate modern vehicles.

Where structure does exist, it is organised by **physics and equipment rather
than by brand**: the cleanest group in the whole solution contains trucks and
large SUVs and nothing else, separated on rollover probability and crash-test
score alone, without the algorithm ever being shown a body style.